In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import re

In [ ]:
df = pd.read_csv("final_dataset.csv")

def tokenize(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)
    return text.split()

# build vocab
vocab = {"<pad>":0, "<unk>":1}
for text in df["symptom_text"]:
    for w in tokenize(text):
        if w not in vocab:
            vocab[w] = len(vocab)

vocab_size = len(vocab)
print("Vocab:", vocab_size)

In [ ]:
MAX_LEN = 20

def encode(text):
    tokens = tokenize(text)
    ids = [vocab.get(t,1) for t in tokens][:MAX_LEN]
    ids += [0]*(MAX_LEN-len(ids))
    return ids

In [ ]:
import ast

all_drugs = sorted(set([d for meds in df["medications"].apply(ast.literal_eval) for d in meds]))
drug2idx = {d:i for i,d in enumerate(all_drugs)}
NUM_DRUGS = len(all_drugs)

class MedDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        ids = torch.tensor(encode(row["symptom_text"]))

        # structured
        age = row["age"]/100
        sex = row["sex"]
        comorb = [1 if c in row["comorbidities"] else 0 for c in ["diabetes","hypertension","asthma"]]
        struct = torch.tensor([age, sex] + comorb, dtype=torch.float)

        # labels
        meds = ast.literal_eval(row["medications"])
        label = torch.zeros(NUM_DRUGS)
        for m in meds:
            if m in drug2idx:
                label[drug2idx[m]] = 1

        return ids, struct, label

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.fc = nn.Linear(64*MAX_LEN + 5, NUM_DRUGS)

    def forward(self, x, struct):
        x = self.emb(x).view(x.size(0), -1)
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class TextCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.conv = nn.Conv1d(64, 128, 3)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(128 + 5, NUM_DRUGS)

    def forward(self, x, struct):
        x = self.emb(x).permute(0,2,1)
        x = self.pool(torch.relu(self.conv(x))).squeeze()
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class BiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(256 + 5, NUM_DRUGS)

    def forward(self, x, struct):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        x = torch.cat([h[0], h[1]], dim=1)
        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)

        self.q = nn.Linear(64, 64)
        self.k = nn.Linear(64, 64)
        self.v = nn.Linear(64, 64)

        self.fc = nn.Linear(64 + 5, NUM_DRUGS)

    def forward(self, x, struct):
        x = self.emb(x)

        Q = self.q(x)
        K = self.k(x)
        V = self.v(x)

        attn = torch.softmax(Q @ K.transpose(-2,-1) / 8, dim=-1)
        x = (attn @ V).mean(dim=1)

        x = torch.cat([x, struct], dim=1)
        return self.fc(x)

In [ ]:
class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, 64)
        self.lstm = nn.LSTM(64, 128, batch_first=True)

        self.struct_fc = nn.Linear(5, 64)

        self.fc = nn.Linear(128 + 64, NUM_DRUGS)

    def forward(self, x, struct):
        x = self.emb(x)
        _, (h, _) = self.lstm(x)
        x = h[-1]

        s = torch.relu(self.struct_fc(struct))

        x = torch.cat([x, s], dim=1)
        return self.fc(x)

In [ ]:
def train(model, loader):
    model = model.cuda()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(3):
        for x, struct, y in loader:
            x, struct, y = x.cuda(), struct.cuda(), y.cuda()

            pred = model(x, struct)
            loss = nn.BCEWithLogitsLoss()(pred, y)

            opt.zero_grad()
            loss.backward()
            opt.step()

        print("Loss:", loss.item())

In [ ]:
dataset = MedDataset(df)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

models = {
    "MLP": MLP(),
    "CNN": TextCNN(),
    "BiLSTM": BiLSTM(),
    "SelfAttention": SelfAttention(),
    "FusionNet": FusionNet()
}

for name, m in models.items():
    print("Training:", name)
    train(m, loader)